# Breakout Strategy Test

Carver's rolling breakout (Strategy 18). The signal is the price's position within a
rolling N-day high/low channel:

    max = rolling max over N,  min = rolling min over N,  mid = (max + min) / 2
    raw = 40 * (p - mid) / (max - min)          # ~[-20, +20]
    smoothed = EWMA(span=N/4)[raw]              # reduce turnover
    signal = (smoothed / forecast_scalar).clip(-2, 2)

A new N-day high drives a positive signal (long); a new low, a negative signal (short).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "sysstrat").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / "data"
print(f"Project root: {project_root}")

from sysstrat.data import load_simple_price_csv
from sysstrat.core import Asset, Capital, FixedRiskSizer
from sysstrat.engine import BacktestRunner, PortfolioRunner
from sysstrat.strategies import BuyAndHoldStrategy, EWMACStrategy, NormalisedTrendStrategy, BreakoutStrategy
from sysstrat.visualization import print_comparison_table

## 1. Load instruments (common period)

Same 5 instruments as before, sliced to their shared date range.

In [ ]:
INSTRUMENTS = {
    "MCFTR":    "MCFTR.csv",
    "RGBITR":   "RGBITR.csv",
    "GLDRUB":   "GLDRUB_TOM.csv",
    "CNYRUB":   "CNYRUB_TOM.csv",
    "USDRUB":   "USDRUB.csv",
}

assets = {
    t: Asset(ticker=t, price_data=load_simple_price_csv(DATA_DIR / f), commission_rate=0.0004, slippage_rate=0.001)
    for t, f in INSTRUMENTS.items()
}
start = max(a.price_data.index.min() for a in assets.values())
end = min(a.price_data.index.max() for a in assets.values())
assets = {t: a.slice(start, end) for t, a in assets.items()}
print(f"Common period: {start.date()} -> {end.date()}")

## 2. Strategies

The breakout rule at two horizons (40 = fast, 160 = slow) vs the other rules.
Each forecast scalar was calibrated on this 5-instrument basket so `avg|signal| ~ 1`.

In [ ]:
CAPITAL = 100_000
capital = Capital(initial_capital=CAPITAL)
sizer = FixedRiskSizer(risk_target=0.20, max_leverage=1.0)

STRATEGIES = {
    "Buy & Hold":      BuyAndHoldStrategy(),
    "EWMAC (16/64)":   EWMACStrategy(),
    "Norm Trend":      NormalisedTrendStrategy(),
    "Breakout (40)":   BreakoutStrategy(horizon=40),
    "Breakout (160)":  BreakoutStrategy(horizon=160, forecast_scalar=11.191),
}

## 3. Equal-weight portfolios (one per strategy)

Each strategy is run on all 5 instruments and combined at equal weight.

In [ ]:
import pandas as pd

rows = []
for name, strategy in STRATEGIES.items():
    reports = {t: BacktestRunner(capital, a, sizer).run(strategy) for t, a in assets.items()}
    port = PortfolioRunner(capital).run(reports)
    m = port.metrics
    rows.append({
        "strategy": name,
        "vol %": round(m.annual_volatility_pct, 2),
        "sharpe": round(m.sharpe_ratio, 2),
        "sortino": round(m.sortino_ratio, 2),
        "max DD %": round(m.max_drawdown_pct, 1),
        "total ret %": round(m.total_return_pct, 1),
    })

print(pd.DataFrame(rows).set_index("strategy").to_string())

## 4. Signal correlation (MCFTR)

Breakout is a different rule family — it should be less correlated with the trend
rules than they are with each other, which is what gives diversification value.

In [ ]:
import numpy as np

d = assets["MCFTR"].price_data.to_frame(name="close")
signals = {
    "EWMAC":     EWMACStrategy().generate_signals(d),
    "NormTrend": NormalisedTrendStrategy().generate_signals(d),
    "Break40":   BreakoutStrategy(horizon=40).generate_signals(d),
    "Break160":  BreakoutStrategy(horizon=160, forecast_scalar=11.191).generate_signals(d),
}

corr = pd.DataFrame({a: {b: round(signals[a].corr(signals[b]), 3) for b in signals} for a in signals})
print(corr)